### Run inference Google

Required inputs:

file_name = "interactions_output_file.parquet" <- parquet file containing all interactions

directory = "images" <- directory containing interaction images

api = "..." <- your own Google api 

In [ ]:
file_name = "interactions_output_file.parquet" 
directory = "images"
api = "YOUR_GOOGLE_API_KEY"

Libraries:

In [ ]:
import os
import time
from pathlib import Path
import pandas as pd
from google import genai
from google.genai import types

In [ ]:
client = genai.Client(api_key=api)

MODELS = [
    "gemini-3-flash-preview",
    #"gemini-3.1-flash-lite-preview",
    #"gemini-3.1-pro-preview"
]

def load_image_part(image_path: str):
    """
    Creates an inline image part for Gemini.
    """
    path = Path(image_path)
    if not path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    suffix = path.suffix.lower()
    mime_map = {
        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
        ".png": "image/png",
        ".webp": "image/webp",
    }
    mime_type = mime_map.get(suffix)
    if mime_type is None:
        raise ValueError(f"Unsupported image type: {suffix}")

    image_bytes = path.read_bytes()
    return types.Part.from_bytes(data=image_bytes, mime_type=mime_type)


def ask_gemini(model_name: str, image_path: str, caption: str, temperature: float = 0.0) -> str:
    """
    Sends one image + one question to Gemini and returns text.
    """
    image_part = load_image_part(image_path)

    response = client.models.generate_content(
        model=model_name,
        contents=caption,
        contents=[
            image_part,
            caption,
        ],
        config=types.GenerateContentConfig(
            temperature=temperature,
        ),
    )

    return (response.text or "").strip()


df = pd.read_parquet(file_name,   
                    engine="pyarrow",
                    dtype_backend="pyarrow")
required_cols = ["fileName", "caption"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

results = []

for idx, row in df.iterrows():
    image_path = row["fileName"]
    caption = row["caption"]
    ground_truth = row["ground_truth"] if "ground_truth" in df.columns else None

    print(f"\nRow {idx}")
    print("Image:", image_path)
    print("Caption:", caption)

    for model_name in MODELS:
        try:
            prediction = ask_gemini(
                model_name=model_name,
                image_path=image_path,
                caption=caption,
                temperature=0.0,   
            )

            result_row = {
                "index": idx,
                "model": model_name,
                "fileName": row["fileName"],
                "caption": caption,
                "prediction": prediction,
                "status": "ok",
            }

            if ground_truth is not None:
                result_row["ground_truth"] = ground_truth

            results.append(result_row)
            print(prediction)
            print(f"[{model_name}] OK")

        except Exception as e:
            result_row = {
                "index": idx,
                "model": model_name,
                "fileName": row["fileName"],
                "caption": caption,
                "prediction": None,
                "status": f"error: {e}",
            }

            if ground_truth is not None:
                result_row["ground_truth"] = ground_truth

            results.append(result_row)

            print(f"[{model_name}] ERROR: {e}")

        time.sleep(0.5)  

results_df = pd.DataFrame(results)
results_df.to_csv("google_inference_results.csv", index=False)

print("\nDone.")
print(results_df.head())